# Notebook 2 — Research Paper Search and Summarization

**Project:** Research Paper Intelligence Assistant  
**Run in:** Google Colab  

This notebook builds a small NLP pipeline from uploaded PDFs: extract text, split pages into overlapping chunks, create a TF-IDF search index, retrieve relevant passages for a question, and generate an extractive summary.

**Important:** TF-IDF is a keyword-based retrieval method, not a generative AI model. The summary selects and reorders sentences already present in the PDF. It can miss nuance, so verify important claims against the original paper and page number.

In [1]:
!pip -q install pymupdf scikit-learn pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 40.7 MB/s eta 0:00:00


## 1. Import libraries

In [2]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import fitz
from google.colab import files
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("NLP search tools are ready.")


NLP search tools are ready.


## 2. Upload papers

In [4]:
uploaded = files.upload()
pdf_paths = [name for name in uploaded if name.lower().endswith(".pdf")]
csv_paths = [name for name in uploaded if name.lower().endswith(".csv")]

if not pdf_paths and not csv_paths:
    raise ValueError("Please upload at least one PDF or CSV file and rerun this cell.")

print(f"Ready to process {len(pdf_paths)} PDF(s) and {len(csv_paths)} CSV(s).")

Saving research_paper_pages.csv to research_paper_pages.csv
Ready to process 0 PDF(s) and 1 CSV(s).


## 3. Extract the text and retain page numbers

In [5]:
def clean_text(text):
    text = str(text).replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

page_records = []
errors = []

# Process PDF files
for pdf_path in pdf_paths:
    try:
        with fitz.open(pdf_path) as pdf:
            for page_number, page in enumerate(pdf, start=1):
                page_text = clean_text(page.get_text("text"))
                if page_text:
                    page_records.append({
                        "paper": Path(pdf_path).name,
                        "page": page_number,
                        "text": page_text,
                        "word_count": len(page_text.split()),
                    })
    except Exception as exc:
        errors.append((pdf_path, str(exc)))

# Process CSV files
for csv_path in csv_paths:
    try:
        df_temp = pd.read_csv(csv_path)
        # Combine all columns of each row into text to index the CSV rows as 'pages'
        for idx, row in df_temp.iterrows():
            row_text = " ".join([f"{col}: {val}" for col, val in row.items() if pd.notna(val)])
            row_text = clean_text(row_text)
            if row_text:
                page_records.append({
                    "paper": Path(csv_path).name,
                    "page": idx + 1,
                    "text": row_text,
                    "word_count": len(row_text.split()),
                })
    except Exception as exc:
        errors.append((csv_path, str(exc)))

pages_df = pd.DataFrame(page_records)
if pages_df.empty:
    raise ValueError("No selectable text found in the uploaded PDF or CSV files.")

print(f"Extracted {len(pages_df)} items/pages in total.")
display(pages_df[["paper", "page", "word_count"]].head())
if errors:
    print("Files with errors:", errors)

Extracted 287421 items/pages in total.


,paper,page,word_count
0,research_paper_pages.csv,1,89
1,research_paper_pages.csv,2,129
2,research_paper_pages.csv,3,177
3,research_paper_pages.csv,4,184
4,research_paper_pages.csv,5,197


## 4. Create overlapping chunks

Long pages are split into smaller passages. The overlap helps reduce the chance of a relevant sentence being cut at a chunk boundary.

In [6]:
def make_chunks(text, chunk_size=180, overlap=40):
    """Create word-based chunks and avoid invalid overlap settings."""
    if chunk_size < 1:
        raise ValueError("chunk_size must be at least 1")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be >= 0 and smaller than chunk_size")
    words = text.split()
    if not words:
        return []
    step = chunk_size - overlap
    return [" ".join(words[start:start + chunk_size])
            for start in range(0, len(words), step)]

chunks = []
for row in pages_df.itertuples(index=False):
    for chunk_number, chunk_text in enumerate(make_chunks(row.text), start=1):
        if chunk_text.strip():
            chunks.append({
                "paper": row.paper,
                "page": int(row.page),
                "chunk": chunk_number,
                "text": chunk_text,
            })

chunks_df = pd.DataFrame(chunks)
if chunks_df.empty:
    raise ValueError("No text chunks were created.")
print(f"Created {len(chunks_df)} text chunks.")
display(chunks_df.head(3))


Created 524701 text chunks.


,paper,page,chunk,text
0,research_paper_pages.csv,1,1,paper: arXiv_scientific_dataset (1).csv page: ...
1,research_paper_pages.csv,2,1,paper: arXiv_scientific_dataset (1).csv page: ...
2,research_paper_pages.csv,3,1,paper: arXiv_scientific_dataset (1).csv page: ...


## 5. Build the TF-IDF search index

TF-IDF gives higher weight to words that are useful for distinguishing one passage from others. Query similarity is calculated with cosine similarity.

In [7]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=50000,
    ngram_range=(1, 2),
)
chunk_matrix = vectorizer.fit_transform(chunks_df["text"].tolist())
print("Index built. Matrix shape:", chunk_matrix.shape)


Index built. Matrix shape: (524701, 50000)


## 6. Search the papers

Change the text in `question` to search for a concept, method, limitation, dataset, or result. Each result includes its source PDF and page number.

In [8]:
def search_papers(question, top_k=5):
    """Return the most relevant text chunks for a keyword-style question."""
    question = question.strip()
    if not question:
        print("Enter a search question first.")
        return pd.DataFrame(columns=["rank", "score", "paper", "page", "text"])

    query_vector = vectorizer.transform([question])
    scores = cosine_similarity(query_vector, chunk_matrix).ravel()
    if not np.any(scores > 0):
        print("No matching terms were found. Try different keywords.")
        return pd.DataFrame(columns=["rank", "score", "paper", "page", "text"])

    best_indices = np.argsort(scores)[::-1][:max(1, top_k)]
    results = chunks_df.iloc[best_indices].copy()
    results.insert(0, "score", scores[best_indices])
    results.insert(0, "rank", range(1, len(results) + 1))
    results = results[results["score"] > 0]
    return results[["rank", "score", "paper", "page", "text"]].reset_index(drop=True)

question = "What machine learning methods and evaluation metrics are used?"
results_df = search_papers(question, top_k=5)
if not results_df.empty:
    display(results_df.style.format({"score": "{:.3f}"}))


,rank,score,paper,page,text
0,1,0.337,research_paper_pages.csv,74751,"to the baseline, largely outperforms OOD detection compared to several posthoc methods on three evaluation metrics: AUROC, AUPR and FPR95. word_count: 154 character_count: 1091"
1,2,0.298,research_paper_pages.csv,257278,"paper: arXiv_scientific_dataset (1).csv page: 257278 text: Understanding the function of individual units in a neural network is an important building block for mechanistic interpretability. This is often done by generating a simple text explanation of the behavior of individual neurons or units. For these explanations to be useful, we must understand how reliable and truthful they are. In this work we unify many existing explanation evaluation methods under one mathematical framework. This allows us to compare existing evaluation metrics, understand the evaluation pipeline with increased clarity and apply existing statistical methods on the evaluation. In addition, we propose two simple sanity checks on the evaluation metrics and show that many commonly used metrics fail these tests and do not change their score after massive changes to the concept labels. Based on our experimental and theoretical results, we propose guidelines that future evaluations should follow and identify a set of reliable evaluation metrics. word_count: 146 character_count: 972"
2,3,0.296,research_paper_pages.csv,136267,"used multi-label datasets, including image and vector datasets. The results demonstrate that our method outperforms state-of-the-art methods in six evaluation metrics. word_count: 155 character_count: 1125"
3,4,0.291,research_paper_pages.csv,93659,"paper: arXiv_scientific_dataset (1).csv page: 93659 text: Machine learning continues to grow in popularity in academia, in industry, and is increasingly used in other fields. However, most of the common metrics used to evaluate even simple binary classification models have shortcomings that are neither immediately obvious nor consistently taught to practitioners. Here we present Interactive Classification Metrics (ICM), an application to visualize and explore the relationships between different evaluation metrics. The user changes the distribution statistics and explores corresponding changes across a suite of evaluation metrics. The interactive, graphical nature of this tool emphasizes the tradeoffs of each metric without the overhead of data wrangling and model training. The goals of this application are: (1) to aid practitioners in the ever-expanding machine learning field to choose the most appropriate evaluation metrics for their classification problem; (2) to promote careful attention to interpretation that is required even in the simplest scenarios like binary classification. Our application is publicly available for free under the MIT license as a Python package on PyPI at https://pypi.org/project/interactive-classification-metrics and on GitHub at https://github.com/davhbrown/interactive_classification_metrics. word_count: 170 character_count: 1283"
4,5,0.283,research_paper_pages.csv,127205,that certain LLMs demonstrate a greater range and diversity of edit operations. Our performance benchmark will be available as a resource for the development of future TS methods and evaluation metrics. word_count: 165 character_count: 1121


## 7. Display one result with a page citation

In [9]:
if not results_df.empty:
    first = results_df.iloc[0]
    print(f"Source: {first['paper']}, page {first['page']}")
    print(f"Similarity score: {first['score']:.3f}\n")
    print(first["text"])
else:
    print("No result to display. Try editing the search question in the previous cell.")


Source: research_paper_pages.csv, page 74751
Similarity score: 0.337

to the baseline, largely outperforms OOD detection compared to several posthoc methods on three evaluation metrics: AUROC, AUPR and FPR95. word_count: 154 character_count: 1091


## 8. Create an extractive summary

This method scores sentences using TF-IDF features and picks high-scoring sentences. It does not write new sentences, so the output should be treated as a reading aid rather than a substitute for reading the paper.

In [10]:
def split_sentences(text):
    # A simple sentence splitter for a beginner-friendly baseline.
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if len(s.split()) >= 6]

def extractive_summary(text, max_sentences=6):
    sentences = split_sentences(text)
    if not sentences:
        return "Not enough readable sentences were extracted."
    if len(sentences) <= max_sentences:
        return " ".join(sentences)

    sentence_vectorizer = TfidfVectorizer(stop_words="english")
    matrix = sentence_vectorizer.fit_transform(sentences)
    # Average TF-IDF importance of each sentence's terms.
    scores = np.asarray(matrix.mean(axis=1)).ravel()
    # Keep the best sentences, but show them in the original reading order.
    chosen_indices = sorted(np.argsort(scores)[-max_sentences:])
    return " ".join(sentences[i] for i in chosen_indices)

# Select a paper to summarize. Change this filename to another uploaded PDF.
selected_paper = pages_df["paper"].iloc[0]
paper_text = "\n".join(pages_df.loc[pages_df["paper"] == selected_paper, "text"].tolist())
summary = extractive_summary(paper_text, max_sentences=6)
print("Paper:", selected_paper)
print("\nEXTRACTIVE SUMMARY\n")
print(summary)


Paper: research_paper_pages.csv

EXTRACTIVE SUMMARY

word_count: 291 character_count: 1856
paper: arXiv_scientific_dataset (1).csv page: 20513 text: The LCLS2 Free Electron Laser FEL will generate xray pulses to beamline
experiments at up to 1Mhz These experimentals will require new ultrahigh rate
UHR detectors that can operate at rates above 100 kHz and generate data
throughputs upwards of 1 TBs a data velocity which requires prohibitively large
investments in storage infrastructure Machine Learning has demonstrated the
potential to digest large datasets to extract relevant insights however current
implementations show latencies that are too high for realtime data reduction
objectives SLAC has endeavored on the creation of a software framework which
translates MLs structures for deployment on Field Programmable Gate Arrays
FPGAs deployed at the Edge of the data chain close to the instrumentation This
framework leverages Xilinxs HLS framework presenting an API modeled after the
open so

## 9. Export search chunks as CSV

In [11]:
output_name = "research_paper_search_chunks.csv"
chunks_df.to_csv(output_name, index=False, encoding="utf-8-sig")
files.download(output_name)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Possible upgrades

1. Replace TF-IDF retrieval with Sentence Transformers embeddings for semantic search.
2. Add a question-answering model and require every answer to cite an extracted passage.
3. Add OCR for scanned papers.
4. Evaluate retrieval quality with a small set of questions and known relevant pages.
5. Deploy the Python app through Streamlit Community Cloud; GitHub Pages alone cannot execute Python server code.